In [0]:
import requests
from bs4 import BeautifulSoup
import random
import pandas as pd

In [0]:
title = 'data engineer'
location = 'india'

list_url =  "https://www.linkedin.com/jobs/search?keywords=Data%20Engineer%20and%20Data%20Anlyst%20and%20Machine%20Learning%20Engineer%20and%20AI%20Engineer%20and%20Generative%20Ai%20Engineer%2&location=india&position=1&pageNum=0"
response = requests.get(list_url)
list_data = response.text
list_soup = BeautifulSoup(list_data, "html.parser")
page_jobs = list_soup.find_all("li")

In [0]:
id_list = []
for job in page_jobs:
    try:
        base_card_div = job.find("div", {"class": "base-card"})
        job_id = base_card_div.get("data-entity-urn").split(":")[3]
        # print(job_id)
        id_list.append(job_id)
    except:
        job_id = None
print(id_list)


In [0]:
# Initialize an empty list to store job information
job_list = []

# Loop through the list of job IDs and get each URL
for job_id in id_list:
    # Construct the URL for each job using the job ID
    job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"

    # Send a GET request to the job URL and parse the reponse
    job_response = requests.get(job_url)
    # print(job_response.status_code)
    job_soup = BeautifulSoup(job_response.text, "html.parser")

    # Create a dictionary to store job details
    job_post = {}

    # Try to extract and store the job title
    try:
        job_post["job_title"] = job_soup.find(
            "h2",
            {
                "class": "top-card-layout__title font-sans text-lg papabear:text-xl font-bold leading-open text-color-text mb-0 topcard__title"
            },
        ).text.strip()
    except:
        job_post["job_title"] = None

    # Try to extract and store the company name
    try:
        job_post["company_name"] = job_soup.find(
            "a", {"class": "topcard__org-name-link topcard__flavor--black-link"}
        ).text.strip()
    except:
        job_post["company_name"] = None

    # Try to extract and store the time posted
    try:
        job_post["time_posted"] = job_soup.find(
            "span", {"class": "posted-time-ago__text topcard__flavor--metadata"}
        ).text.strip()
    except:
        job_post["time_posted"] = None

    # Try to extract and store the number of applicants
    try:
        job_post["num_applicants"] = job_soup.find(
            "span",
            {
                "class": "num-applicants__caption topcard__flavor--metadata topcard__flavor--bullet"
            },
        ).text.strip()
    except:
        job_post["num_applicants"] = None

    # Try to extract and store the number of applicants
    try:
        job_post["seniority_level"] = job_soup.find(
            "span",
            {
                "class": "description__job-criteria-text description__job-criteria-text--criteria"
            },
        ).text.strip()
    except:
        job_post["seniority_level"] = None

    # Try to extract and store the number of applicants
    try:
        job_post["employment_type"] = job_soup.find_all(
            "span",
            {
                "class": "description__job-criteria-text description__job-criteria-text--criteria"
            },
        ).text.strip()
    except:
        job_post["employment_type"] = None

    # Append the job details to the job_list
    job_list.append(job_post)

In [0]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import random
import time
import re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}


class LinkedInScraper:
    def __init__(self, keywords, location):
        self.keywords = keywords
        self.location = location
        self.base_url = (
            f"https://www.linkedin.com/jobs/search?"
            f"keywords={keywords}&location={location}&position=1&pageNum=0"
        )

    def get_job_ids(self):
        """Extract job IDs from search result page"""
        response = requests.get(self.base_url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")
        jobs = soup.find_all("li")

        job_ids = []
        for job in jobs:
            try:
                base_card_div = job.find("div", {"class": "base-card"})
                job_id = base_card_div.get("data-entity-urn").split(":")[3]
                job_ids.append(job_id)
            except Exception:
                continue
        return job_ids

    def extract_responsibilities_and_skills(self, soup):
        """Extract responsibilities, skills, and raw description"""
        description_div = soup.select_one("div.show-more-less-html__markup")
        responsibilities, skills, raw_description = [], [], None

        if description_div:
            raw_description = description_div.get_text(" ", strip=True)
            current_section = None

            for elem in description_div.children:
                # Detect section headers
                if elem.name == "p":
                    text = elem.get_text(strip=True)
                    if "responsibilit" in text.lower():
                        current_section = "responsibilities"
                    elif "skill" in text.lower():
                        current_section = "skills"

                # Collect bullet points
                elif elem.name == "ul":
                    bullets = [li.get_text(strip=True) for li in elem.find_all("li")]
                    if current_section == "responsibilities":
                        responsibilities.extend(bullets)
                    elif current_section == "skills":
                        skills.extend(bullets)

        # Fallback: split description into sentences if no bullets found
        if not responsibilities and not skills and raw_description:
            sentences = [s.strip() for s in re.split(r"[.!?]", raw_description) if s.strip()]
            responsibilities = sentences

        return responsibilities, skills, raw_description

    def get_job_details(self, job_id):
        """Extract job details from job posting page"""
        job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
        response = requests.get(job_url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")
        job_post = {}

        def safe_extract(selector, multiple=False):
            try:
                if multiple:
                    return [s.text.strip() for s in soup.select(selector)]
                return soup.select_one(selector).text.strip()
            except Exception:
                return None

        # Base job info
        job_post["job_id"] = job_id
        job_post["job_title"] = safe_extract("h2.top-card-layout__title")
        job_post["company_name"] = safe_extract("a.topcard__org-name-link")
        job_post["time_posted"] = safe_extract("span.posted-time-ago__text")
        job_post["num_applicants"] = safe_extract("span.num-applicants__caption")

        # === LOCATION FIELDS ===
        job_post["search_location"] = self.location

        loc = None
        loc_selectors = [
            "span.topcard__flavor--bullet",
            "span.topcard__flavor",
            "span.job-location",
            "div.top-card-layout__entity-info span"
        ]
        for sel in loc_selectors:
            loc = safe_extract(sel)
            if loc:
                break

        job_post["job_posting_location"] = None  # temporary, update later

        # Criteria
        criteria = safe_extract("span.description__job-criteria-text", multiple=True)
        if criteria and len(criteria) >= 4:
            job_post["seniority_level"] = criteria[0]
            job_post["employment_type"] = criteria[1]
            job_post["job_function"] = criteria[2]
            job_post["industries"] = criteria[3]
        else:
            job_post["seniority_level"] = None
            job_post["employment_type"] = None
            job_post["job_function"] = None
            job_post["industries"] = None

        # Responsibilities & skills
        responsibilities, skills, raw_desc = self.extract_responsibilities_and_skills(soup)
        job_post["responsibilities"] = responsibilities
        job_post["skills"] = skills
        job_post["raw_description"] = raw_desc

        # If location not found earlier, check description
        if not loc and raw_desc:
            m = re.search(r"(?:Work location|Role location|Location)[:\s]*([^.\n\r]+)", raw_desc, re.IGNORECASE)
            if m:
                loc = m.group(1).strip()

        job_post["job_posting_location"] = loc
        return job_post

    def scrape_jobs(self):
        """Main method to scrape jobs"""
        job_ids = self.get_job_ids()
        job_list = []

        for job_id in job_ids:
            details = self.get_job_details(job_id)
            job_list.append(details)
            time.sleep(random.uniform(1, 3))  # polite scraping

        return job_list


In [0]:
import pandas as pd
from scraper import LinkedInScraper

if __name__ == "__main__":
    keywords = "Data Engineer OR Data Analyst OR Machine Learning Engineer"
    locations = ["India", "USA", "Australia"]

    all_jobs = []

    for loc in locations:
        print(f"\n🔍 Scraping jobs for: {loc}")
        scraper = LinkedInScraper(keywords=keywords, location=loc)

        try:
            jobs = scraper.scrape_jobs()
            print(f"✅ Extracted {len(jobs)} jobs for {loc}")

            # Add search_location explicitly
            for job in jobs:
                job["search_location"] = loc

            all_jobs.extend(jobs)

        except Exception as e:
            print(f"❌ Failed scraping for {loc}: {e}")

    # Convert to DataFrame
    df = pd.DataFrame(all_jobs)

    print(f"\n📊 Final dataset shape: {df.shape}")
    print(df.head())

    # Save as JSON
    df.to_json("jobs_multilocation.json", orient="records", lines=True)
    print("💾 Saved jobs_multilocation.json")


In [0]:
# scraper = LinkedInScraper("data engineer", "india")
# print("Base URL:", scraper.base_url)

from scraper import LinkedInScraper

locations = ["India", "USA", "Australia"]
keyword = "data engineer"

for loc in locations:
    scraper = LinkedInScraper(keyword, loc)
    print(f"🔍 {loc} → {scraper.base_url}")


In [0]:
job_ids = scraper.get_job_ids()
print("Extracted Job IDs:", job_ids)
print("Total Jobs Found:", len(job_ids))

if not job_ids:
    print(scraper.base_url)
    response = requests.get(scraper.base_url, headers=HEADERS)
    print(response.text[:1000])

In [0]:
if job_ids:
    sample_job_id = job_ids[0]
    print("Testing with Job ID:", sample_job_id)

    job_details = scraper.get_job_details(sample_job_id)
    print("Job Details Extracted:")
    print(job_details)

In [0]:
if job_ids:
    sample_job_id = job_ids[0]
    print("Testing with Job ID:", sample_job_id)

    job_details = scraper.get_job_details(sample_job_id)
    print("Job Details Extracted:")
    print(job_details)

In [0]:
def safe_extract(selector, multiple=False):
    try:
        if multiple:
            values = [s.text.strip() for s in soup.select(selector)]
            print(f"[DEBUG] Extracted list for {selector}: {values}")
            return values
        value = soup.select_one(selector).text.strip()
        print(f"[DEBUG] Extracted single for {selector}: {value}")
        return value
    except Exception as e:
        print(f"[DEBUG] Failed to extract {selector}: {e}")
        return None

In [0]:
import pandas as pd
from scraper import LinkedInScraper

if __name__ == "__main__":
    keywords = "Data Engineer"
    location = "India"

    scraper = LinkedInScraper(keywords=keywords, location=location)

    # Get job IDs
    job_ids = scraper.get_job_ids()
    print("Job IDs found:", job_ids[:5])  # show first 5 for debug

    if job_ids:
        # Pick the first job ID for testing
        test_job_id = job_ids[0]
        print(f"\n🔍 Checking details for Job ID: {test_job_id}")

        job_details = scraper.get_job_details(test_job_id)

        # Display only a few key fields for clarity
        print("\n✅ Extracted Job Details:")
        for k in ["job_id", "job_title", "company_name", "search_location", "job_posting_location"]:
            print(f"{k}: {job_details.get(k)}")

        # If you want to see full job dict
        # import pprint; pprint.pprint(job_details)
    else:
        print("❌ No job IDs found.")


In [0]:
from scraper import LinkedInScraper

if __name__ == "__main__":
    # Pick one location just for initialization (needed for constructor)
    scraper = LinkedInScraper(keywords="Data Engineer", location="India")

    # 👇 Test a single known Job ID
    job_id = "4282147859"   # McDonald's Data Engineer II
    job_details = scraper.get_job_details(job_id)

    print("\n✅ Extracted Job Details:")
    for k, v in job_details.items():
        print(f"{k}: {v}")